# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

1. NEAR_MISS-DECLINING - avg_position(8-20), 75th-percentile decline cutoff.
2. NEAR_MISS_STALE - avg_position(8-20), stale( >= 200 days), not declining.
3. NEAR_MISS_STALE_DECLINING- avg_position, both conditions hold.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df['trend_pct_flipped'] = -df['trend_pct']
lower = df['trend_pct_flipped'].quantile(0.01)
upper = df['trend_pct_flipped'].quantile(0.99)

df['trend_pct_dampened'] = df['trend_pct_flipped'].clip(lower, upper)

In [3]:
import numpy as np

# value + rank components (computed across the full 30,000 rows)
value = df['cpc'] * df['search_volume']
declining_rank = df['trend_pct_dampened'].abs().rank(pct=True)
staleness_rank = df['days_since_last_update'].rank(pct=True)

# masks
near_miss_mask = df['avg_position'].between(8, 20)
decline_threshold = df.loc[near_miss_mask, 'trend_pct_dampened'].quantile(0.75)  # ≈61.5
stale_threshold = 200

declining_only_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] < stale_threshold)
stale_only_mask = near_miss_mask & (df['days_since_last_update'] >= stale_threshold) & (df['trend_pct_dampened'] < decline_threshold)
stale_declining_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] >= stale_threshold)

# scores (unconditional, np.select will pick the right one per row)
score_declining = value * declining_rank
score_stale = value * staleness_rank
score_stale_declining = value * (declining_rank + staleness_rank)

conditions = [stale_declining_mask, declining_only_mask, stale_only_mask]
score_choices = [score_stale_declining, score_declining, score_stale]
reason_choices = ['NEAR_MISS_STALE_DECLINING', 'NEAR_MISS_DECLINING', 'NEAR_MISS_STALE']

df['baseline_score'] = np.select(conditions, score_choices, default=0)
df['reason_code'] = np.select(conditions, reason_choices, default=None)

In [4]:
df['reason_code'].value_counts()

reason_code
NEAR_MISS_DECLINING          2444
NEAR_MISS_STALE                11
NEAR_MISS_STALE_DECLINING       7
Name: count, dtype: int64

In [5]:
flagged = df[df['reason_code'].notna()].sort_values('baseline_score', ascending=False)

In [6]:
import os
os.makedirs('../outputs', exist_ok=True)
flagged[['content_id', 'baseline_score', 'reason_code', 'avg_position', 'cpc', 'search_volume', 'days_since_last_update']].to_csv('../outputs/baseline_action_score.csv', index=False)

In [7]:
flagged.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,trend_pct_flipped,trend_pct_dampened,baseline_score,reason_code
24994,content_f3f0a4ce87b4,client_3fdba35f04,4400.0,1.00,HIGH,8.32,keyword article,transactional,2723.0,16848.0,...,13.59,0.0,good,striking,down,-67.0,67.0,67.0,26017.101458,NEAR_MISS_DECLINING
9217,content_12e48d4b449d,client_8527a891e2,27100.0,0.04,LOW,0.93,keyword article,commercial,3579.0,21187.0,...,50.00,0.0,low,striking,down,-100.0,100.0,100.0,23207.083778,NEAR_MISS_DECLINING
29458,content_bc1fc1980b74,client_19581e27de,14800.0,0.93,HIGH,1.86,keyword article,informational,NaN,NaN,...,0.00,0.0,low,striking,down,-63.2,63.2,63.2,18525.437848,NEAR_MISS_DECLINING
4639,content_271062e724db,client_f369cb89fc,8100.0,1.00,HIGH,2.71,keyword article,transactional,2526.0,15274.0,...,12.50,0.0,moderate,striking,down,-75.3,75.3,75.3,16994.043195,NEAR_MISS_DECLINING
3752,content_e7eb94e121b9,client_6208ef0f77,22200.0,0.76,HIGH,0.79,keyword article,commercial,5032.0,33055.0,...,0.00,0.0,moderate,striking,down,-83.9,83.9,83.9,14556.236848,NEAR_MISS_DECLINING
26642,content_df34036c8115,client_a88a7902cb,14800.0,0.99,HIGH,1.08,keyword article,transactional,3719.0,25432.0,...,100.00,0.0,low,striking,down,-83.3,83.3,83.3,13206.981211,NEAR_MISS_DECLINING
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,...,0.00,0.0,low,striking,down,-100.0,100.0,100.0,11548.352121,NEAR_MISS_STALE_DECLINING
5118,content_c861e30f2f7a,client_6208ef0f77,22200.0,0.84,HIGH,0.57,keyword article,commercial,4471.0,29022.0,...,3.70,0.0,low,page_1,down,-88.2,88.2,88.2,10785.285886,NEAR_MISS_DECLINING
23095,content_3bd65ea2d52c,client_7f2253d7e2,2400.0,0.31,LOW,5.71,keyword article,informational,2888.0,18484.0,...,33.33,0.0,moderate,page_1,down,-69.4,69.4,69.4,9988.337442,NEAR_MISS_DECLINING
28333,content_811b2bcff32f,client_19581e27de,12100.0,0.92,HIGH,0.81,keyword article,transactional,NaN,NaN,...,8.00,0.0,low,striking,down,-85.7,85.7,85.7,8226.181272,NEAR_MISS_DECLINING


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
flagged.head(20)[['content_id', 'baseline_score', 'reason_code', 'avg_position', 'cpc', 'search_volume', 'days_since_last_update']]

,content_id,baseline_score,reason_code,avg_position,cpc,search_volume,days_since_last_update
24994,content_f3f0a4ce87b4,26017.101458,NEAR_MISS_DECLINING,11.4,8.32,4400.0,26
9217,content_12e48d4b449d,23207.083778,NEAR_MISS_DECLINING,17.0,0.93,27100.0,20
29458,content_bc1fc1980b74,18525.437848,NEAR_MISS_DECLINING,16.1,1.86,14800.0,20
4639,content_271062e724db,16994.043195,NEAR_MISS_DECLINING,15.7,2.71,8100.0,20
3752,content_e7eb94e121b9,14556.236848,NEAR_MISS_DECLINING,15.6,0.79,22200.0,104
26642,content_df34036c8115,13206.981211,NEAR_MISS_DECLINING,16.3,1.08,14800.0,8
1659,content_bbca724138f2,11548.352121,NEAR_MISS_STALE_DECLINING,12.1,3.76,1600.0,236
5118,content_c861e30f2f7a,10785.285886,NEAR_MISS_DECLINING,8.4,0.57,22200.0,104
23095,content_3bd65ea2d52c,9988.337442,NEAR_MISS_DECLINING,8.2,5.71,2400.0,20
28333,content_811b2bcff32f,8226.181272,NEAR_MISS_DECLINING,11.1,0.81,12100.0,22


1. content_f3f0a4ce87b4 — score 26,017, NEAR_MISS_DECLINING
Position 11.4, cpc $8.32, search_volume 4,400, updated 26 days ago.

Action: Refresh content — likely needs a content quality or relevance pass, since it's not stale (touched recently) but still declining.
Confidence note: Medium confidence — the score here is carried by a high cpc (8.32) rather than volume, suggesting real commercial intent behind this page. Still, cpc alone doesn't confirm clicks are actually happening — worth checking the decline isn't just a seasonal dip before committing editor time.
What would make it wrong: If the decline is seasonal rather than a real quality problem, refreshing it wouldn't help — worth checking whether this content_type/intent has a seasonal pattern before acting.

2. content_12e48d4b449d — score 23,207, NEAR_MISS_DECLINING
Position 17.0, cpc $0.93, search_volume 27,100.

Action: Same reason code, but this page's high score comes almost entirely from search_volume, not cpc — worth a human check on whether that traffic actually converts.
Confidence note: Lower confidence — low cpc alongside very high search_volume suggests the topic may get looked at but not bought into. Score qualifies the page, but commercial value is questionable.
What would make it wrong: Low cpc could mean commercial intent isn't really there — refreshing a page nobody's buying anything from may not be worth editor time even with impressive volume.

3. content_bbca724138f2 — score 11,548, NEAR_MISS_STALE_DECLINING
Position 12.1, cpc $3.76, search_volume 1,600, 236 days since update.

Action: Prime refresh candidate — genuinely old content that's also actively declining.
Confidence note: High confidence — this is one of the few pages hitting both decline and staleness (236 days untouched), the exact double-signal the labeling logic was designed to catch. Low risk of false positive.
What would make it wrong: Little downside risk here — dual-signal cases like this are the strongest picks in the queue.

4. content_3f7dbbd55f0c — score 3,288, NEAR_MISS_DECLINING
Position 10.2, cpc $0.25, search_volume 14,800, updated 104 days ago.

Action: Lower-priority refresh — flagged mainly on volume, not commercial value.
Confidence note: Low confidence — cpc of $0.25 is tiny, meaning this page likely has traffic but almost no commercial value. Ranking it this high purely on search_volume is a concrete example of the score's value-dominance limitation.
What would make it wrong: If cpc this low reflects genuinely low-value intent (e.g. informational-only searches), this page may not deserve editor time despite qualifying on paper.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:
Several top-20 rows are flagged mainly on search_volume with very low cpc (e.g. content_12e48d4b449d: cpc $0.93, volume 27,100; content_3f7dbbd55f0c: cpc $0.25, volume 14,800). High traffic doesn't guarantee commercial value — a page nobody's buying from may not be worth editor time even at a high score. This reflects a real limitation in the current formula: once a page clears the decline threshold (75th percentile of trend_pct_dampened within the near-miss zone), ranking is driven almost entirely by cpc × search_volume rather than decline severity. Two pages can share a reason code but represent very different underlying problems — one because it's declining hard, another mostly because it happens to have a big audience. A future iteration could weight decline severity more heavily even after the qualifying threshold, rather than treating it as a pass/fail gate.

Leakage check:
The baseline score and reason codes are built from avg_position, trend_pct_dampened, days_since_last_update, cpc, and search_volume — all five were already verified in the Week 3 data contract as knowable before the prediction moment, and none are label-derived. No product flags (e.g. health_score, existing FlyRank tags) were used as inputs — this is a hand-written rule, not a reproduction of an existing system's decision. No future or overlapping windows were touched — the same 90-day aggregate columns used throughout, none of the excluded 30-day/leakage columns (impressions_last_30d, trend_pct, trend_direction, etc.) were referenced anywhere in the scoring logic.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.